# 02 — Exploratory Data Analysis

Visualize class imbalance, feature distributions, correlations, and transaction-amount patterns.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

In [ ]:
df = pd.read_csv('../data/raw/creditcard.csv')

## Class Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
counts = df['Class'].value_counts()
axes[0].bar(['Legitimate (0)', 'Fraud (1)'], counts.values, color=['steelblue', 'tomato'])
axes[0].set_title('Transaction Count by Class')
axes[0].set_ylabel('Count')
for i, v in enumerate(counts.values):
    axes[0].text(i, v + 500, f'{v:,}', ha='center')

axes[1].pie(counts.values, labels=['Legitimate', 'Fraud'], autopct='%1.4f%%',
            colors=['steelblue', 'tomato'], startangle=90)
axes[1].set_title('Class Proportion')
plt.tight_layout()
plt.show()

## Transaction Amount: Fraud vs Legitimate

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for label, color, name in [(0, 'steelblue', 'Legitimate'), (1, 'tomato', 'Fraud')]:
    subset = df[df['Class'] == label]['Amount']
    axes[0].hist(subset, bins=60, alpha=0.6, label=name, color=color, density=True)
axes[0].set_xlabel('Amount')
axes[0].set_title('Amount Distribution (log-scale x)')
axes[0].set_xscale('symlog')
axes[0].legend()

df.boxplot(column='Amount', by='Class', ax=axes[1])
axes[1].set_title('Amount by Class')
axes[1].set_xlabel('Class (0=Legit, 1=Fraud)')
plt.suptitle('')
plt.tight_layout()
plt.show()

print(df.groupby('Class')['Amount'].describe())

## Transactions Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.scatter(df[df['Class']==0]['Time'], df[df['Class']==0]['Amount'],
           alpha=0.1, s=1, color='steelblue', label='Legitimate')
ax.scatter(df[df['Class']==1]['Time'], df[df['Class']==1]['Amount'],
           alpha=0.6, s=8, color='tomato', label='Fraud')
ax.set_xlabel('Time (seconds)')
ax.set_ylabel('Amount')
ax.set_title('Transactions Over Time')
ax.legend()
plt.tight_layout()
plt.show()

## PCA Feature Distributions (V1–V10)

In [ ]:
features = [f'V{i}' for i in range(1, 11)]
fig, axes = plt.subplots(2, 5, figsize=(20, 7))
axes = axes.flatten()
for i, feat in enumerate(features):
    axes[i].hist(df[df['Class']==0][feat], bins=50, alpha=0.6, label='Legit', color='steelblue', density=True)
    axes[i].hist(df[df['Class']==1][feat], bins=50, alpha=0.7, label='Fraud', color='tomato', density=True)
    axes[i].set_title(feat)
    axes[i].legend(fontsize=8)
plt.suptitle('PCA Feature Distributions — Fraud vs Legitimate', y=1.01)
plt.tight_layout()
plt.show()

## Correlation Heatmap (top features with Class)

In [ ]:
corr = df.corr()['Class'].drop('Class').abs().sort_values(ascending=False)
top_features = corr.head(15).index.tolist()

fig, ax = plt.subplots(figsize=(10, 6))
corr_matrix = df[top_features + ['Class']].corr()
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Correlation Matrix — Top 15 Features')
plt.tight_layout()
plt.show()

print('\nTop 10 features correlated with Class:')
print(corr.head(10))